In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Soft Margin Tradeoff

This notebook gives a conceptual view of how the regularization parameter $C$ changes the tradeoff between a wide margin and strict handling of outliers. The geometry is illustrative rather than the exact optimizer output.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


In [ ]:
LEFT = np.array([[-2.5, -0.8], [-2.2, 0.5], [-1.8, -1.4], [-1.5, 0.9], [-0.9, -0.4]])
RIGHT = np.array([[1.3, -0.2], [1.9, 0.8], [2.1, -1.1], [2.7, 0.1], [2.4, 1.2]])
OUTLIER = np.array([[-0.15, 0.35]])

def plot_soft_margin(log10_c=0.0):
    c_value = 10 ** log10_c
    boundary_x = 0.12 * np.tanh(log10_c) - 0.08 * np.tanh(1.4 * log10_c)
    margin_half = max(0.55, 1.45 - 0.22 * log10_c)

    def margin_score(points, label):
        signed = label * (points[:, 0] - boundary_x)
        return signed < margin_half

    left_violations = margin_score(LEFT, -1)
    right_violations = margin_score(RIGHT, 1)
    outlier_violation = margin_score(OUTLIER, 1)[0]
    total_violations = int(left_violations.sum() + right_violations.sum() + outlier_violation)

    fig, ax = plt.subplots(figsize=(7.4, 5.8))
    ax.axvspan(boundary_x - margin_half, boundary_x + margin_half, color='#fde68a', alpha=0.35, label='Margin band')
    ax.axvline(boundary_x, color='#0f172a', linewidth=2.2, label='Decision boundary')
    ax.axvline(boundary_x - margin_half, color='#64748b', linestyle='--', linewidth=1.5)
    ax.axvline(boundary_x + margin_half, color='#64748b', linestyle='--', linewidth=1.5)

    ax.scatter(LEFT[:, 0], LEFT[:, 1], c='#2563eb', s=54, label='Class -1')
    ax.scatter(RIGHT[:, 0], RIGHT[:, 1], c='#dc2626', s=54, label='Class +1')
    ax.scatter(OUTLIER[:, 0], OUTLIER[:, 1], c='#dc2626', s=86, marker='X', label='Outlier')

    for points, mask in [(LEFT, left_violations), (RIGHT, right_violations)]:
        if mask.any():
            ax.scatter(points[mask, 0], points[mask, 1], facecolors='none', edgecolors='#111827', s=180, linewidths=1.8)
    if outlier_violation:
        ax.scatter(OUTLIER[:, 0], OUTLIER[:, 1], facecolors='none', edgecolors='#111827', s=240, linewidths=1.8)

    ax.text(-3.1, 2.45, f'C = {c_value:.3f}', fontsize=12, weight='bold')
    ax.text(-3.1, 2.1, f'Conceptual margin width = {2 * margin_half:.2f}', fontsize=11)
    ax.text(-3.1, 1.75, f'Points inside margin or misclassified = {total_violations}', fontsize=11)
    ax.text(-3.1, 1.35, 'Small C: wider margin, more tolerance', fontsize=10, color='#334155')
    ax.text(-3.1, 1.05, 'Large C: narrower margin, stronger penalty', fontsize=10, color='#334155')

    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-2.1, 2.8)
    ax.grid(alpha=0.18)
    ax.legend(loc='lower right')
    ax.set_title('Conceptual Soft-Margin SVM Tradeoff')
    plt.show()

widgets.interact(
    plot_soft_margin,
    log10_c=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='log10 C')
);
